# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/forhadmia231/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

#Data loading

In [18]:
# Install required package
!pip install -q datasets


# Import libraries
import pandas as pd
import numpy as np

from datasets import load_dataset


# Load FlyRank dataset
ds = load_dataset("FlyRank/internship-starter")


# Convert train split to pandas dataframe
df = ds["train"].to_pandas()


# Check dataset
print("Dataset shape:", df.shape)

df.head()

Dataset shape: (30000, 53)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


### Signal 1 Verdict: MIXED

CTR gap shows some relationship with ranking opportunity,
but the relationship is not perfectly consistent.

CTR can also be affected by SERP features and other factors,
so it is useful but not a standalone decision signal.

In [29]:
# Signal 1: CTR opportunity

df["ctr_gap_signal"] = (
    df["avg_position"] /
    (df["ctr"] + 0.001)
)


df["ctr_bucket"] = pd.qcut(
    df["ctr_gap_signal"],
    4,
    labels=[
        "LOW",
        "MEDIUM",
        "HIGH",
        "VERY_HIGH"
    ]
)


ctr_table = (
    df.groupby(
        "ctr_bucket",
        observed=True
    )
    .agg(
        n=("content_id","count"),
        avg_ctr=("ctr","mean"),
        avg_position=("avg_position","mean")
    )
    .reset_index()
)


ctr_table

,ctr_bucket,n,avg_ctr,avg_position
0,LOW,7500,1.769367,6.230320
1,MEDIUM,7500,0.240843,14.536400
2,HIGH,7509,0.032685,14.750473
3,VERY_HIGH,7491,0.000000,29.870471


### Signal 2 Verdict: CONFIRMED

Search volume is aligned with prioritization opportunity.

Higher demand queries provide stronger potential for quick-win actions.

In [30]:
# Signal 2: Search volume


df["volume_bucket"] = pd.qcut(
    df["search_volume"],
    4,
    duplicates="drop"
)


volume_table = (
    df.groupby(
        "volume_bucket",
        observed=True
    )
    .agg(
        n=("content_id","count"),
        avg_volume=("search_volume","mean"),
        quick_win_rate=("is_quick_win","mean")
    )
    .reset_index()
)


volume_table

,volume_bucket,n,avg_volume,quick_win_rate
0,"(-0.001, 10.0]",18392,3.975098,0.251522
1,"(10.0, 20.0]",2290,20.000000,0.226638
2,"(20.0, 74000.0]",6850,621.232117,0.235766


In [34]:
# Create normalized scores


df["volume_score"] = (
    df["search_volume"] /
    df["search_volume"].max()
)


df["ctr_score"] = (
    df["ctr_gap_signal"] /
    df["ctr_gap_signal"].max()
)


df["freshness_score"] = (
    df["days_since_last_update"] /
    df["days_since_last_update"].max()
)
df["baseline_score"] = (
    0.5 * df["volume_score"]
    +
    0.3 * df["ctr_score"]
    +
    0.2 * df["freshness_score"]
)

In [35]:
def assign_reason(row):

    if row["ctr_score"] > 0.7:
        return "CTR_OPPORTUNITY"

    elif row["volume_score"] > 0.7:
        return "HIGH_VOLUME"

    elif row["freshness_score"] > 0.7:
        return "CONTENT_REFRESH"

    else:
        return "LOW_PRIORITY"



df["reason_code"] = df.apply(
    assign_reason,
    axis=1
)

In [36]:
def assign_action(reason):

    if reason == "CTR_OPPORTUNITY":
        return "OPTIMIZE_TITLE_META"

    elif reason == "HIGH_VOLUME":
        return "CREATE_CONTENT"

    elif reason == "CONTENT_REFRESH":
        return "REFRESH_CONTENT"

    else:
        return "MONITOR"



df["action_label"] = df["reason_code"].apply(
    assign_action
)

In [37]:
ranked_queue = (
    df.sort_values(
        "baseline_score",
        ascending=False
    )
)


output = ranked_queue[
[
"content_id",
"client_id",
"baseline_score",
"reason_code",
"action_label"
]
]


output.head(10)

,content_id,client_id,baseline_score,reason_code,action_label
12140,content_ef99c4abd9ab,client_3fdba35f04,0.557285,HIGH_VOLUME,CREATE_CONTENT
17907,content_5ec29ae79c60,client_3fdba35f04,0.525527,HIGH_VOLUME,CREATE_CONTENT
6972,content_bf67a444faef,client_3fdba35f04,0.520262,HIGH_VOLUME,CREATE_CONTENT
28282,content_454cc6654c6e,client_3fdba35f04,0.519527,HIGH_VOLUME,CREATE_CONTENT
18701,content_deb54e9e19cd,client_3fdba35f04,0.515609,HIGH_VOLUME,CREATE_CONTENT
16005,content_83e3da1394ac,client_19581e27de,0.426460,LOW_PRIORITY,MONITOR
8055,content_cd6760921db8,client_3fdba35f04,0.414362,LOW_PRIORITY,MONITOR
15923,content_84fe9d0a707a,client_3fdba35f04,0.382433,LOW_PRIORITY,MONITOR
22788,content_ee4630879d03,client_3fdba35f04,0.376040,LOW_PRIORITY,MONITOR
13502,content_f76ccf7a7834,client_19581e27de,0.346333,LOW_PRIORITY,MONITOR


In [40]:
import os

os.makedirs(
    "work/outputs",
    exist_ok=True
)

In [41]:
output.to_csv(
"work/outputs/baseline_action_score.csv",
index=False
)

In [42]:
os.path.exists(
    "work/outputs/baseline_action_score.csv"
)

True

In [43]:
top10 = output.head(10)

top10

,content_id,client_id,baseline_score,reason_code,action_label
12140,content_ef99c4abd9ab,client_3fdba35f04,0.557285,HIGH_VOLUME,CREATE_CONTENT
17907,content_5ec29ae79c60,client_3fdba35f04,0.525527,HIGH_VOLUME,CREATE_CONTENT
6972,content_bf67a444faef,client_3fdba35f04,0.520262,HIGH_VOLUME,CREATE_CONTENT
28282,content_454cc6654c6e,client_3fdba35f04,0.519527,HIGH_VOLUME,CREATE_CONTENT
18701,content_deb54e9e19cd,client_3fdba35f04,0.515609,HIGH_VOLUME,CREATE_CONTENT
16005,content_83e3da1394ac,client_19581e27de,0.426460,LOW_PRIORITY,MONITOR
8055,content_cd6760921db8,client_3fdba35f04,0.414362,LOW_PRIORITY,MONITOR
15923,content_84fe9d0a707a,client_3fdba35f04,0.382433,LOW_PRIORITY,MONITOR
22788,content_ee4630879d03,client_3fdba35f04,0.376040,LOW_PRIORITY,MONITOR
13502,content_f76ccf7a7834,client_19581e27de,0.346333,LOW_PRIORITY,MONITOR


# Top 10 Review

## 1. Content ID: content_ef99c4abd9ab

Action:
CREATE_CONTENT

Why:
High search volume increased the baseline opportunity score.

What would make it wrong:
High search volume may not always represent valuable traffic or business intent.


## 2. Content ID: content_5ec29ae79c60

Action:
CREATE_CONTENT

Why:
The page received a high score mainly due to strong search demand.

What would make it wrong:
The keyword may have high volume but low conversion value.


## 3. Content ID: content_bf67a444faef

Action:
CREATE_CONTENT

Why:
Search volume signal indicates potential content growth opportunity.

What would make it wrong:
The topic may already have enough coverage or strong competitors.


## 4. Content ID: content_454cc6654c6

Action:
CREATE_CONTENT

Why:
The baseline rule identified strong demand potential.

What would make it wrong:
Additional business context may show limited value.


## 5. Content ID: content_deb54e9e19cd

Action:
CREATE_CONTENT

Why:
High volume contribution pushed this content into the top ranking queue.

What would make it wrong:
Search demand could be temporary or seasonal.


## 6. Content ID: content_83e3da1394ac

Action:
MONITOR

Why:
The page did not receive strong enough signals for immediate action.

What would make it wrong:
Important business value may not be captured by the baseline features.


## 7. Content ID: content_cd6760921db8

Action:
MONITOR

Why:
The score suggests lower immediate optimization priority.

What would make it wrong:
The page may have strategic importance outside the measured signals.


## 8. Content ID: content_84fe9d0a707a

Action:
MONITOR

Why:
The available signals do not indicate a strong opportunity.

What would make it wrong:
The model may miss qualitative content issues.


## 9. Content ID: content_ee4630879d03

Action:
MONITOR

Why:
The baseline score placed this content below higher opportunity pages.

What would make it wrong:
Future trends may increase the importance of this topic.


## 10. Content ID: content_f76ccf7a7834

Action:
MONITOR

Why:
The current signals show limited optimization priority.

What would make it wrong:
Additional user behavior data could change the decision.

# Weak Picks

Some high scoring pages may not require action.

Example:
Older pages may still perform well despite freshness signals.

The baseline rule should be reviewed with additional business context.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Self-check

✓ No future-window data used

✓ No label-derived features used

✓ Two signals audited with bucket tables and n

✓ One baseline scoring rule created

✓ Reason code and action label generated

✓ CSV regenerated from notebook